In [1]:
import sys
sys.path.append('/data/Aldhani/cv_fields/code/decode/')
sys.path.append('/data/Aldhani/cv_fields/code/')
sys.path.append('/data/Aldhani/cv_fields/code/smallholder-fields/code/')

In [2]:
import os
import cv2
import glob
import warnings
import numpy as np
import time
from tqdm import tqdm
from osgeo import gdal
from osgeo import ogr
import matplotlib.pyplot as plt
from sklearn.metrics import matthews_corrcoef
import pandas as pd

import mxnet as mx
from mxnet import nd, gpu, gluon, autograd, npx, image
from mxnet.gluon.data import DataLoader
from mxnet.gluon.data.vision import transforms
from mxnet.gluon.loss import Loss

In [3]:
from decode.FracTAL_ResUNet.nn.loss.mtsk_loss import *
from decode.FracTAL_ResUNet.models.heads.head_cmtsk import *
from decode.FracTAL_ResUNet.models.semanticsegmentation.FracTAL_ResUNet_features import *
from decode.FracTAL_ResUNet.models.semanticsegmentation.FracTAL_ResUNet import FracTAL_ResUNet_cmtsk

from helpers_model import *

In [4]:
# download pretrained model from mozambique dataset (Rufin et al. 2026)
import urllib.request
os.makedirs("../model", exist_ok=True)
if not os.path.exists("../model/airbus_mozambique.params"):
    urllib.request.urlretrieve(
        "https://zenodo.org/records/17531366/files/model.params?download=1",
        "../model/airbus_mozambique.params")

In [ ]:
# choose fine-tuned model, alternatively use '../model/airbus_mozambique.params'
model = '../model/fractal-resunet_finetune-v01'

# store outputs
write=True

# define inputs
img_path = '../sample_data/'
prd_path = f'../preds/{os.path.basename(model)}/'

print(prd_path)
if not os.path.exists('../preds/'): 
    os.mkdir('../preds/')
if not os.path.exists(prd_path): 
    os.mkdir(prd_path)

in_files = sorted(glob.glob(img_path + '*.tif'))
len(in_files)

../preds/fractal-resunet_finetune-v01/


1

In [6]:
# images of 4096x4096 chipped into subsets
# subset size and step size
# 50% overlap to avoid edge effects
size=1024
step=512

# D6nf32
depth=6
NClasses=1
nfilters_init=32
linear_norm = False
norm = 'none'

# load params
net = FracTAL_ResUNet_cmtsk(nfilters_init=nfilters_init, NClasses=NClasses,depth=depth)
net.load_parameters(f'{model}.params')
net.collect_params().reset_ctx(gpu(0))

depth:= 0, nfilters: 32, nheads::8, widths::1
depth:= 1, nfilters: 64, nheads::16, widths::1
depth:= 2, nfilters: 128, nheads::32, widths::1
depth:= 3, nfilters: 256, nheads::64, widths::1
depth:= 4, nfilters: 512, nheads::128, widths::1
depth:= 5, nfilters: 1024, nheads::256, widths::1
depth:= 6, nfilters: 512, nheads::256, widths::1
depth:= 7, nfilters: 256, nheads::128, widths::1
depth:= 8, nfilters: 128, nheads::64, widths::1
depth:= 9, nfilters: 64, nheads::32, widths::1
depth:= 10, nfilters: 32, nheads::16, widths::1


[19:10:38] /work/mxnet/src/base.cc:79: cuDNN lib mismatch: linked-against version 8600 != compiled-against version 8500.  Set MXNET_CUDNN_LIB_CHECKING=0 to quiet this warning.


In [9]:
# executes inference pipeline iterates across tiles and subwindows
# test time ~28 sec per 4096x4096 image on GPU (NVIDIA A30 24GB)
for in_file in in_files:
  tic = time.time()
  out_file = prd_path + os.path.basename(in_file)[:-4]+'_preds.tif'
  # open
  if not os.path.isfile(out_file):
    input = open_tif_rgb_tc(in_file)
    
    # write output?
    if write:
      # create memory copy of ds
      mem_ds = create_mem_ds(in_file, 3, dtype=gdal.GDT_Int16)

    for y in np.arange(0, input.shape[1], step):   
      for x in np.arange(0, input.shape[2], step): 
        
        in_tns = input[None,:,y:y+size,x:x+size]
        
        # check if in_tns all zero?
        if nd.sum(in_tns[:,0,:,:])>0:
          # predict
          preds = net(in_tns)

          if write:
            # write outputs to bands
            for b in range(3):
              bnd = np.squeeze(preds[b]).asnumpy()
              bnd = (bnd*10000).astype(int)
              
              if (y==0) & (x==0):
                bnd = bnd[:int(step*1.5),:int(step*1.5)]
                mem_ds.GetRasterBand(b+1).WriteArray(bnd, xoff=x, yoff=y)

              if (y==0) & (x>0):
                bnd = bnd[:int(step*1.5),int(step*0.5):int(step*1.5)]
                mem_ds.GetRasterBand(b+1).WriteArray(bnd, xoff=x+step*0.5, yoff=y)

              if (y>0) & (x==0):
                bnd = bnd[int(step*0.5):int(step*1.5),:int(step*1.5)]
                mem_ds.GetRasterBand(b+1).WriteArray(bnd, xoff=x, yoff=y+step*0.5)

              if (y>0) & (x>0) & (x<input.shape[2]-step) & (y<input.shape[1]-step):
                bnd = bnd[int(step*0.5):int(step*1.5),int(step*0.5):int(step*1.5)]
                mem_ds.GetRasterBand(b+1).WriteArray(bnd, xoff=x+step*0.5, yoff=y+step*0.5)
              
              # last row
              if (x==input.shape[2]-step) & (y!=input.shape[1]-step):
                bnd = bnd[int(step*0.5):,int(step*0.5):int(step*1.5)]
                mem_ds.GetRasterBand(b+1).WriteArray(bnd, xoff=x+step*0.5, yoff=y+step*0.5)
              
              # last col
              if (x!=input.shape[2]-step) & (y==input.shape[1]-step):
                bnd = bnd[int(step*0.5):int(step*1.5),int(step*0.5):]
                mem_ds.GetRasterBand(b+1).WriteArray(bnd, xoff=x+step*0.5, yoff=y+step*0.5)
              
              # last tile
              if (x==input.shape[2]-step) & (y==input.shape[1]-step):
                bnd = bnd[int(step*0.5):,int(step*0.5):]
                mem_ds.GetRasterBand(b+1).WriteArray(bnd, xoff=x+step*0.5, yoff=y+step*0.5)

    # create physical copy of ds
    copy_mem_ds(out_file, mem_ds)
    print(out_file)
      
    print(time.time()-tic)

../preds/fractal-resunet_finetune-v01/mozambique_tile_32636_00020_-0281_20230917_preds.tif
7.24756121635437


ERROR 4: Attempt to create new tiff file `../preds/fractal-resunet_finetune-v01/mozambique_tile_32636_00020_-0281_20230917_preds.tif' failed: ../preds/fractal-resunet_finetune-v01/mozambique_tile_32636_00020_-0281_20230917_preds.tif: No such file or directory
